# 10 Museums domain — Wikidata multihop benchmark generator

Clean replacement for the museums part of `10_paintings_museums.ipynb`. It generates RU/EN tasks, English-only clean constraints, full WDQS gold sets, ASK validators, and JSONL records with the same `BenchmarkExample` structure used by the other domains.

In [1]:

# ============================================================
# Common setup for Wikidata benchmark domain notebooks
# ============================================================
from pathlib import Path
import json
import math
import random
import re
import time
import shutil
from collections import Counter, defaultdict
from dataclasses import asdict, fields
from typing import Any, Dict, List, Optional, Sequence, Tuple

# Load shared benchmark helpers. The second path makes this notebook runnable
# in the ChatGPT sandbox; the first path is the normal project-local path.
if "BenchmarkExample" not in globals():
    _helper_candidates = [Path("common_helpers.py"), Path("/mnt/data/common_helpers.py")]
    for _p in _helper_candidates:
        if _p.exists():
            exec(_p.read_text(encoding="utf-8"), globals())
            break
    else:
        raise FileNotFoundError("common_helpers.py not found. Put it next to this notebook.")

DOMAIN_OUTPUT_DIR = Path(OUT_DIR) / "domain_outputs"
DOMAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_BENCHMARK_FIELD_ORDER = [
    "id", "domain", "complexity", "query_text_ru", "constraints", "requested_count",
    "gold_answer_qids", "gold_answer_labels_ru", "sparql_query", "created_at",
    "query_text_en", "gold_answer_labels_en", "is_advanced", "template_id",
    "template_family", "gold_truncated", "ask_validator_sparql", "local_validator",
    "gold_collection_meta", "gold_answer_imdb_ids", "gold_answer_imdb_titles",
]
# Hard guard: every domain JSONL must have exactly the same top-level schema and field order
# as the final JSONL files from the other domains (cinema/geo/software/etc.).
BENCHMARK_FIELD_ORDER = EXPECTED_BENCHMARK_FIELD_ORDER
assert [f.name for f in fields(BenchmarkExample)] == BENCHMARK_FIELD_ORDER, [f.name for f in fields(BenchmarkExample)]

# Optional hard cross-check against an existing final-domain JSONL. If one of these
# files is present next to the notebook, its first row must have exactly the same
# physical top-level field order. This catches silent schema drift before generation.
REFERENCE_JSONL_CANDIDATES = [
    Path("cinema.jsonl"), Path("/mnt/data/cinema.jsonl"),
    Path("geo_international.jsonl"), Path("/mnt/data/geo_international.jsonl"),
    Path("software.jsonl"), Path("/mnt/data/software.jsonl"),
]

def _first_jsonl_keys(path: Path) -> Optional[List[str]]:
    try:
        if not path.exists():
            return None
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    obj = json.loads(line)
                    return list(obj.keys()) if isinstance(obj, dict) else None
    except Exception as e:
        print(f"[WARN] could not inspect reference JSONL {path}: {e}")
    return None

for _ref_jsonl in REFERENCE_JSONL_CANDIDATES:
    _ref_keys = _first_jsonl_keys(_ref_jsonl)
    if _ref_keys:
        if _ref_keys != BENCHMARK_FIELD_ORDER:
            raise AssertionError({"reference_jsonl": str(_ref_jsonl), "expected": BENCHMARK_FIELD_ORDER, "actual": _ref_keys})
        print(f"✅ JSONL field order matches reference: {_ref_jsonl}")
        break
QID_RE_LOCAL = re.compile(r"^Q\d+$")


def _good_qid(x: Any) -> bool:
    return isinstance(x, str) and bool(QID_RE_LOCAL.fullmatch(x.strip()))


def _good_label(x: Any) -> bool:
    if x is None:
        return False
    s = str(x).strip()
    return bool(s) and not QID_RE_LOCAL.fullmatch(s)


def _safe_int(x: Any, default: Optional[int] = None) -> Optional[int]:
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return default
        return int(float(str(x).strip()))
    except Exception:
        return default


def _parse_wikidata_year(x: Any) -> Optional[int]:
    """Parse WDQS date/year strings such as 1889-01-01T00:00:00Z."""
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    m = re.search(r"([+-]?\d{3,4})", s)
    if not m:
        return None
    y = _safe_int(m.group(1))
    if y is None or y < 1000 or y > 2026:
        return None
    return y


def _year_bucket(y: Optional[int]) -> Tuple[Optional[str], Optional[int], Optional[int], Optional[str], Optional[str]]:
    """Stable, human-readable date buckets used both in query text and constraints."""
    if y is None:
        return None, None, None, None, None
    if y <= 1599:
        return "1400_1599", 1400, 1599, "1400–1599", "1400–1599"
    if y <= 1699:
        return "1600_1699", 1600, 1699, "1600–1699", "1600–1699"
    if y <= 1799:
        return "1700_1799", 1700, 1799, "1700–1799", "1700–1799"
    if y <= 1849:
        return "1800_1849", 1800, 1849, "1800–1849", "1800–1849"
    if y <= 1899:
        return "1850_1899", 1850, 1899, "1850–1899", "1850–1899"
    if y <= 1949:
        return "1900_1949", 1900, 1949, "1900–1949", "1900–1949"
    return "1950_2026", 1950, 2026, "1950–2026", "1950–2026"


def _dedupe_preserve_order(xs: Sequence[Any]) -> List[Any]:
    seen = set()
    out = []
    for x in xs:
        key = json.dumps(x, ensure_ascii=False, sort_keys=True) if isinstance(x, (dict, list)) else str(x)
        if key not in seen:
            seen.add(key)
            out.append(x)
    return out


def _dedupe_lines(lines: Sequence[str]) -> List[str]:
    return [x for x in _dedupe_preserve_order([str(l).strip() for l in lines if str(l).strip()])]


def _entity_label_cache_key(qid: str) -> str:
    return f"wd_labels_ru_en_v3_{qid}"


def get_entity_labels_ru_en(qids: Sequence[str]) -> Dict[str, Dict[str, str]]:
    """Fetch labels via wbgetentities. Returns {qid: {'en': ..., 'ru': ...}} with caching."""
    qids = [q for q in _dedupe_preserve_order([str(q).strip() for q in qids or []]) if _good_qid(q)]
    out: Dict[str, Dict[str, str]] = {}
    missing: List[str] = []
    cache = getattr(wd, "cache", None)

    for q in qids:
        cached = cache.get(_entity_label_cache_key(q)) if cache is not None else None
        if isinstance(cached, dict) and (_good_label(cached.get("en")) or _good_label(cached.get("ru"))):
            en = str(cached.get("en") or cached.get("ru") or "").strip()
            ru = str(cached.get("ru") or cached.get("en") or "").strip()
            out[q] = {"en": en, "ru": ru}
        else:
            missing.append(q)

    for i in range(0, len(missing), 50):
        chunk = missing[i:i+50]
        if not chunk:
            continue
        params = {
            "action": "wbgetentities",
            "ids": "|".join(chunk),
            "props": "labels",
            "languages": "en|ru",
            "format": "json",
        }
        last_err = None
        for attempt in range(4):
            try:
                if hasattr(wd, "_sleep_if_needed"):
                    wd._sleep_if_needed()
                resp = requests.get(WIKI_API, params=params, headers={"User-Agent": USER_AGENT}, timeout=30)
                if hasattr(wd, "_last_request_ts"):
                    wd._last_request_ts = time.time()
                if resp.status_code in (429, 500, 502, 503, 504):
                    last_err = RuntimeError(f"Wikidata API transient HTTP {resp.status_code}")
                    time.sleep(min(20, 2 ** attempt) + random.random())
                    continue
                resp.raise_for_status()
                data = resp.json()
                entities = (data or {}).get("entities", {}) or {}
                for q in chunk:
                    labels = (entities.get(q) or {}).get("labels", {}) or {}
                    en = ((labels.get("en") or {}).get("value") or "").strip()
                    ru = ((labels.get("ru") or {}).get("value") or "").strip()
                    if not _good_label(en) and _good_label(ru):
                        en = ru
                    if not _good_label(ru) and _good_label(en):
                        ru = en
                    if _good_label(en) or _good_label(ru):
                        out[q] = {"en": en, "ru": ru}
                        if cache is not None:
                            try:
                                cache.set(_entity_label_cache_key(q), out[q])
                            except Exception:
                                pass
                break
            except Exception as e:
                last_err = e
                if attempt == 3:
                    print(f"[WARN] label fetch failed for chunk {chunk[:3]}...: {last_err}")
                time.sleep(min(20, 2 ** attempt) + random.random())
    return out


def add_ru_en_labels(df, qid_fields: Sequence[str]):
    """For every `<field>_qid` column, add `<field>_en` and `<field>_ru`."""
    if df is None or len(df) == 0:
        return df
    qids = []
    for field in qid_fields:
        col = f"{field}_qid"
        if col in df.columns:
            qids.extend([q for q in df[col].dropna().astype(str).tolist() if _good_qid(q)])
    labels = get_entity_labels_ru_en(qids)
    out = df.copy()
    for field in qid_fields:
        col = f"{field}_qid"
        if col not in out.columns:
            out[col] = None
        out[f"{field}_en"] = out[col].map(lambda q: (labels.get(str(q)) or {}).get("en") if _good_qid(str(q)) else None)
        out[f"{field}_ru"] = out[col].map(lambda q: (labels.get(str(q)) or {}).get("ru") if _good_qid(str(q)) else None)
    return out


def _read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    out.append(obj)
            except Exception as e:
                print(f"[WARN] bad JSONL line {line_no} in {path}: {e}")
    return out


def _append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _write_json(path: Path, obj: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def _record_key(record: Dict[str, Any]) -> str:
    payload = {
        "domain": record.get("domain"),
        "complexity": record.get("complexity"),
        "template_id": record.get("template_id"),
        "constraints": record.get("constraints"),
        "query_text_en": record.get("query_text_en"),
    }
    return json.dumps(payload, ensure_ascii=False, sort_keys=True)


def _example_to_record(ex: BenchmarkExample) -> Dict[str, Any]:
    raw = asdict(ex)
    # Make the physical JSONL field order exactly match BenchmarkExample/common_helpers.
    return {k: raw.get(k) for k in BENCHMARK_FIELD_ORDER}


def _schema_is_exact(record: Dict[str, Any]) -> bool:
    return list(record.keys()) == BENCHMARK_FIELD_ORDER




_CYRILLIC_RE = re.compile(r"[А-Яа-яЁё]")


def _constraint_value_has_qid(v: Any) -> bool:
    if isinstance(v, str):
        return bool(QID_RE_LOCAL.fullmatch(v.strip()))
    if isinstance(v, list):
        return any(_constraint_value_has_qid(x) for x in v)
    if isinstance(v, dict):
        return any(_constraint_value_has_qid(x) for x in v.values())
    return False


def _constraint_value_has_cyrillic(v: Any) -> bool:
    if isinstance(v, str):
        return bool(_CYRILLIC_RE.search(v))
    if isinstance(v, list):
        return any(_constraint_value_has_cyrillic(x) for x in v)
    if isinstance(v, dict):
        return any(_constraint_value_has_cyrillic(x) for x in v.values())
    return False


def _constraint_qids_are_valid(v: Any) -> bool:
    if isinstance(v, str):
        return _good_qid(v)
    if isinstance(v, list):
        return all(_constraint_qids_are_valid(x) for x in v)
    if isinstance(v, dict):
        return all(_constraint_qids_are_valid(x) for x in v.values())
    return v is None


def _archive_existing_jsonl(path: Path, suffix: str = "bak") -> Optional[Path]:
    """Archive an existing generated JSONL before a clean regeneration."""
    path = Path(path)
    if not path.exists():
        return None
    ts = _now_iso().replace(":", "").replace("-", "").replace(".", "_").replace("Z", "Z")
    archived = path.with_suffix(path.suffix + f".{suffix}_{ts}")
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(path), str(archived))
    print(f"Archived previous output: {archived}")
    return archived


def validate_jsonl_exact_format(path: Path, expected_fields: Sequence[str] = BENCHMARK_FIELD_ORDER) -> pd.DataFrame:
    """Validate exact shared JSONL schema/order plus high-value benchmark invariants."""
    rows = _read_jsonl(path)
    problems = []
    for i, rec in enumerate(rows, start=1):
        keys = list(rec.keys())
        rid = rec.get("id")
        if keys != list(expected_fields):
            problems.append({
                "line": i,
                "id": rid,
                "problem": "field_order_or_field_set_mismatch",
                "actual_keys": keys,
            })
        if not isinstance(rid, str) or not re.search(r"_l[1-5]_\d{4}$", rid):
            problems.append({"line": i, "id": rid, "problem": "id_format_mismatch"})
        if rec.get("complexity") not in {"L1", "L2", "L3", "L4", "L5"}:
            problems.append({"line": i, "id": rid, "problem": "bad_complexity"})
        for key in ("gold_answer_qids", "gold_answer_labels_ru", "gold_answer_labels_en", "gold_answer_imdb_ids", "gold_answer_imdb_titles"):
            if not isinstance(rec.get(key), list):
                problems.append({"line": i, "id": rid, "problem": f"{key}_not_list"})
        qids = rec.get("gold_answer_qids", [])
        if len(qids) != len(rec.get("gold_answer_labels_ru", [])) or len(qids) != len(rec.get("gold_answer_labels_en", [])):
            problems.append({"line": i, "id": rid, "problem": "gold_qids_labels_length_mismatch"})
        if len(qids) != len(set(qids)):
            problems.append({"line": i, "id": rid, "problem": "duplicate_gold_qids"})
        if any(not _good_qid(q) for q in qids):
            problems.append({"line": i, "id": rid, "problem": "bad_gold_qid"})
        if not isinstance(rec.get("constraints"), dict):
            problems.append({"line": i, "id": rid, "problem": "constraints_not_dict"})
        else:
            if _constraint_value_has_qid(rec["constraints"]):
                problems.append({"line": i, "id": rid, "problem": "qid_leaked_into_constraints"})
            if _constraint_value_has_cyrillic(rec["constraints"]):
                problems.append({"line": i, "id": rid, "problem": "cyrillic_leaked_into_constraints"})
        meta = rec.get("gold_collection_meta")
        if not isinstance(meta, dict):
            problems.append({"line": i, "id": rid, "problem": "gold_collection_meta_not_dict"})
        else:
            if meta.get("gold_returned") != len(qids):
                problems.append({"line": i, "id": rid, "problem": "meta_gold_returned_mismatch"})
            c_qids = meta.get("constraint_qids")
            if not isinstance(c_qids, dict):
                problems.append({"line": i, "id": rid, "problem": "missing_constraint_qids"})
            elif not _constraint_qids_are_valid(c_qids):
                problems.append({"line": i, "id": rid, "problem": "bad_constraint_qids"})
        if rec.get("complexity") in {"L3", "L4", "L5"} and rec.get("is_advanced") is not True:
            problems.append({"line": i, "id": rid, "problem": "advanced_level_has_is_advanced_false"})
        if rec.get("complexity") in {"L1", "L2"} and rec.get("is_advanced") is not False:
            problems.append({"line": i, "id": rid, "problem": "basic_level_has_is_advanced_true"})
        if rec.get("gold_truncated") is not False:
            problems.append({"line": i, "id": rid, "problem": "gold_truncated_not_false"})
        if not isinstance(rec.get("sparql_query"), str) or "SELECT" not in rec.get("sparql_query", ""):
            problems.append({"line": i, "id": rid, "problem": "bad_sparql_query"})
        if not isinstance(rec.get("ask_validator_sparql"), str) or "ASK" not in rec.get("ask_validator_sparql", ""):
            problems.append({"line": i, "id": rid, "problem": "bad_ask_validator_sparql"})
    return pd.DataFrame(problems)

def _now_iso() -> str:
    return utc_now_z() if "utc_now_z" in globals() else dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Museums generator core

In [2]:

# ============================================================
# Museums domain: clean WDQS-backed multihop generator
# ============================================================
MUSEUMS_DOMAIN = "museums"
Q_MUSEUM = "Q33506"
MUSEUMS_OUTPUT_PATH = DOMAIN_OUTPUT_DIR / "museums.jsonl"
MUSEUMS_AUDIT_PATH = DOMAIN_OUTPUT_DIR / "museums.audit.json"

MUSEUMS_TARGET_PER_LEVEL = {
    "L1": 20,
    "L2": 22,
    "L3": 24,
    "L4": 22,
    "L5": 22,
}

MUSEUMS_REQUESTED_COUNT = {
    "L1": 5,
    "L2": 5,
    "L3": 4,
    "L4": 3,
    "L5": 3,
}

MUSEUMS_MAX_GOLD_BY_LEVEL = {
    # Deliberately conservative caps: very broad classes/locations are usually
    # low-value benchmark tasks and can hide noisy Wikidata taxonomy edges.
    "L1": 250,
    "L2": 200,
    "L3": 150,
    "L4": 100,
    "L5": 80,
}
MUSEUMS_MIN_GOLD_HEADROOM = 2
MUSEUMS_SEED_LIMIT = 7500
MUSEUMS_MAX_ATTEMPTS_PER_LEVEL = 3500



# Quality filters for museum type labels/classes that are formally present in Wikidata
# but produce noisy or unnatural benchmark tasks.
MUSEUM_BAD_TYPE_QIDS = {
    "Q58632302",  # specialised museum in Finland: taxonomy leaks non-Finnish bunker classes
    "Q3867560",   # Italian national museum: includes many archaeological sites/churches/palaces
    "Q60731643",  # local museum in Finland: too country-specific and awkward as a type
    "Q112132527", # historical park museum: too many historical sites/parks
    "Q115154345", # local authority museum: administrative status, very broad/noisy
    "Q124830213", # museum of a public entity: administrative status, noisy
    "Q115154402", # independent museum: governance status, not a useful topical type
}
MUSEUM_BAD_TYPE_LABEL_PATTERNS = [
    re.compile(r"\bin Finland\b", re.I),
    re.compile(r"^Italian national museum$", re.I),
    re.compile(r"^local authority museum$", re.I),
    re.compile(r"^museum of a public entity$", re.I),
    re.compile(r"^historical park museum$", re.I),
    re.compile(r"^independent museum$", re.I),
]
MUSEUM_GOOD_TYPE_LABEL_PATTERNS = [
    # Keep topical/recognisable museum classes; reject bare administrative/status classes.
    re.compile(r"\bmuseum\b", re.I),
    re.compile(r"\bkunsthalle\b", re.I),
    re.compile(r"\bcinematheque\b", re.I),
    re.compile(r"\bsculpture garden\b", re.I),
    re.compile(r"\bheritage (center|centre|railway)\b", re.I),
    re.compile(r"\bfilm archive\b", re.I),
]
MUSEUM_TYPE_RU_OVERRIDES = {
    "Bible museum": "музей Библии",
    "museum ship": "корабль-музей",
    "aviation museum": "авиационный музей",
    "kunsthalle": "кунстхалле",
    "artist museum": "музей художника",
    "archaeological museum": "археологический музей",
    "archaeological artifact museum": "музей археологических артефактов",
    "art museum": "художественный музей",
    "history museum": "исторический музей",
    "military museum": "военный музей",
    "maritime museum": "морской музей",
    "science museum": "научный музей",
    "technology museum": "технический музей",
    "transport museum": "музей транспорта",
    "working life museum": "музей трудовой истории",
    "local history museum": "краеведческий музей",
    "natural history museum": "музей естественной истории",
}

def _museum_type_is_bad(qid: Any = None, label_en: Any = None) -> bool:
    q = str(qid or "").strip()
    lab = str(label_en or "").strip()
    if q in MUSEUM_BAD_TYPE_QIDS:
        return True
    if any(p.search(lab) for p in MUSEUM_BAD_TYPE_LABEL_PATTERNS):
        return True
    if lab and not any(p.search(lab) for p in MUSEUM_GOOD_TYPE_LABEL_PATTERNS):
        return True
    return False

def _museum_type_ru_label(label_en: str, label_ru: str) -> str:
    en = str(label_en or "").strip()
    ru = str(label_ru or "").strip()
    if en in MUSEUM_TYPE_RU_OVERRIDES:
        return MUSEUM_TYPE_RU_OVERRIDES[en]
    return ru or en

def _ru_museum_count_phrase(k: int) -> str:
    k = int(k)
    if k % 10 in (2, 3, 4) and k % 100 not in (12, 13, 14):
        return f"{k} музея"
    return f"{k} музеев"

def _museum_date_filter(y1: Optional[int], y2: Optional[int]) -> List[str]:
    if y1 is None or y2 is None:
        return []
    return [
        "?item wdt:P571 ?inceptionDate .",
        "BIND(YEAR(?inceptionDate) AS ?inceptionYear) .",
        f"FILTER(?inceptionYear >= {int(y1)} && ?inceptionYear <= {int(y2)}) .",
    ]


MUSEUM_FIELD_BUILDERS = {
    "type": lambda q: [f"?item wdt:P31/wdt:P279* wd:{q} ."],
    "country": lambda q: [f"?item wdt:P17 wd:{q} ."],
    "city": lambda q: [f"?item wdt:P131+ wd:{q} ."],
    "continent": lambda q: ["?item wdt:P17 ?countryForContinent .", f"?countryForContinent wdt:P30 wd:{q} ."],
    "operator": lambda q: [f"?item wdt:P137 wd:{q} ."],
}

MUSEUM_FIELD_CONSTRAINT_NAMES = {
    "type": "museum_type",
    "country": "country",
    "city": "located_in",
    "continent": "continent",
    "operator": "operator",
}


def _museum_fragment(field: str, en: str, ru: str) -> Tuple[str, str]:
    if field == "type":
        ru_label = _museum_type_ru_label(en, ru)
        return (f"относящихся к типу «{ru_label}»", f'classified as "{en}"')
    return {
        "country": (f"расположенных в стране {ru}", f"located in {en}"),
        "city": (f"расположенных в административной единице «{ru}»", f"located in {en}"),
        "continent": (f"расположенных в странах континента {ru}", f"located in countries in {en}"),
        "operator": (f"оператором которых является «{ru}»", f"operated by {en}"),
    }[field]


def _museum_query_text(k: int, fragments_ru: Sequence[str], fragments_en: Sequence[str]) -> Tuple[str, str]:
    count_ru = _ru_museum_count_phrase(k)
    if fragments_ru:
        ru = f"Назови {count_ru}, " + ", ".join(fragments_ru) + "."
    else:
        ru = f"Назови {count_ru}."
    if fragments_en:
        en = f"Name {int(k)} museums " + " and ".join(fragments_en) + "."
    else:
        en = f"Name {int(k)} museums."
    return ru, en


def _build_museums_gold_sparql(where_lines: Sequence[str], limit: int) -> str:
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_MUSEUM} .
      {where}
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
    }}
    ORDER BY LCASE(STR(?itemLabelEn))
    LIMIT {int(limit)}
    """.strip()


def _build_museums_ask(where_lines: Sequence[str]) -> str:
    where = "\n      ".join(_dedupe_lines(where_lines))
    return f"""
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?item)
      ?item wdt:P31/wdt:P279* wd:{Q_MUSEUM} .
      {where}
    }}
    """.strip()


def _run_museums_gold_query(spec: Dict[str, Any], gold_limit: int):
    wdqs_limit = int(gold_limit) + 1
    sparql = _build_museums_gold_sparql(spec["where_lines"], wdqs_limit)
    rows = rows_from_select(wd.sparql_select(sparql))
    items = []
    dropped_no_qid = 0
    dropped_no_en = 0
    label_sources = Counter()
    seen = set()
    for r in rows:
        qid = uri_to_qid(r.get("item", ""))
        en = (r.get("itemLabelEn") or "").strip()
        ru = (r.get("itemLabelRu") or "").strip()
        if not _good_qid(qid):
            dropped_no_qid += 1
            continue
        if not _good_label(en):
            dropped_no_en += 1
            continue
        if not _good_label(ru):
            ru = en
            label_sources["en_fallback_for_ru"] += 1
        else:
            label_sources["ru_label"] += 1
        if qid not in seen:
            seen.add(qid)
            items.append((qid, ru, en))
    # Sentinel logic: if WDQS returns the extra row, reject the candidate as incomplete
    # even if a rare duplicated/label edge leaves <= gold_limit unique QIDs after parsing.
    truncated = len(rows) >= wdqs_limit or len(items) > int(gold_limit)
    return sparql, items[:int(gold_limit)], truncated, {
        "wdqs_candidate_limit": wdqs_limit,
        "rows_returned_by_wdqs": len(rows),
        "gold_returned_before_limits": len(items),
        "dropped_no_qid_count": dropped_no_qid,
        "dropped_no_en_label_count": dropped_no_en,
        "label_sources": dict(label_sources),
        "gold_may_be_incomplete_due_to_wdqs_limit": bool(truncated),
    }


def _museum_local_validator(spec: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "type": "none_wdqs_only",
        "source": "Wikidata Query Service",
        "applies_after": "ask_validator_sparql",
        "filters": spec.get("constraints", {}),
        "label_matching_used": False,
        "note": "All constraints for this museums task are represented in the WDQS ASK validator; no external local validator is required.",
    }


def _finalize_museum_spec(spec: Dict[str, Any], idx: int, require_complete: bool = True) -> Optional[BenchmarkExample]:
    level = spec["complexity"]
    k = int(spec["requested_count"])
    max_gold = int(MUSEUMS_MAX_GOLD_BY_LEVEL[level])
    sparql, gold, truncated, meta0 = _run_museums_gold_query(spec, max_gold)
    if require_complete and truncated:
        return None
    if len(gold) < k + MUSEUMS_MIN_GOLD_HEADROOM:
        return None
    if len(gold) > max_gold:
        return None
    ask = _build_museums_ask(spec["where_lines"])
    meta = {
        "source": "wikidata_sparql",
        **meta0,
        "constraints_are_wdqs_only": True,
        "gold_limit": max_gold,
        "gold_returned": len(gold),
        "gold_total_before_limit": meta0["gold_returned_before_limits"],
        "gold_truncated_by_local_limit": bool(truncated),
        "template_id": spec["template_id"],
        "template_family": spec["template_family"],
        "constraint_qids": spec.get("constraint_qids", {}),
    }
    return BenchmarkExample(
        id=f"museums_{level.lower()}_{idx:04d}",
        domain=MUSEUMS_DOMAIN,
        complexity=level,
        query_text_ru=spec["query_text_ru"],
        constraints=spec["constraints"],
        requested_count=k,
        gold_answer_qids=[q for q, _, _ in gold],
        gold_answer_labels_ru=[ru for _, ru, _ in gold],
        sparql_query=sparql,
        created_at=_now_iso(),
        query_text_en=spec["query_text_en"],
        gold_answer_labels_en=[en for _, _, en in gold],
        is_advanced=level in {"L3", "L4", "L5"},
        template_id=spec["template_id"],
        template_family=spec["template_family"],
        gold_truncated=bool(truncated),
        ask_validator_sparql=ask,
        local_validator=_museum_local_validator(spec),
        gold_collection_meta=meta,
    )


## Seed pool and template registry

In [3]:

# ============================================================
# Museums seed pool and candidate registry
# ============================================================
def build_museums_seed_pool(limit: int = MUSEUMS_SEED_LIMIT):
    """Broad pool of museums and linked facts used only for candidate discovery."""
    sparql = f"""
    SELECT DISTINCT ?item ?type ?country ?city ?continent ?operator ?inception WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_MUSEUM} .
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{
        ?item wdt:P31 ?type .
        ?type wdt:P279* wd:{Q_MUSEUM} .
        FILTER(?type != wd:{Q_MUSEUM})
      }}
      OPTIONAL {{
        ?item wdt:P17 ?country .
        OPTIONAL {{ ?country wdt:P30 ?continent . }}
      }}
      OPTIONAL {{ ?item wdt:P131 ?city . }}
      OPTIONAL {{ ?item wdt:P137 ?operator . }}
      OPTIONAL {{ ?item wdt:P571 ?inception . }}
    }}
    LIMIT {int(limit)}
    """
    rows = rows_from_select(wd.sparql_select(sparql))
    data = []
    for r in rows:
        item_qid = uri_to_qid(r.get("item", ""))
        if not _good_qid(item_qid):
            continue
        y = _parse_wikidata_year(r.get("inception"))
        yb, y1, y2, yru, yen = _year_bucket(y)
        data.append({
            "item_qid": item_qid,
            "type_qid": uri_to_qid(r.get("type", "")),
            "country_qid": uri_to_qid(r.get("country", "")),
            "city_qid": uri_to_qid(r.get("city", "")),
            "continent_qid": uri_to_qid(r.get("continent", "")),
            "operator_qid": uri_to_qid(r.get("operator", "")),
            "year": y,
            "year_bucket": yb,
            "year_from": y1,
            "year_to": y2,
            "year_ru": yru,
            "year_en": yen,
        })
    label_fields = ["item", "type", "country", "city", "continent", "operator"]
    if not data:
        cols = ["item_qid"] + [f"{f}_qid" for f in label_fields if f != "item"] + ["year", "year_bucket", "year_from", "year_to", "year_ru", "year_en"]
        cols += [f"{f}_{lang}" for f in label_fields for lang in ("en", "ru")]
        return pd.DataFrame(columns=_dedupe_preserve_order(cols))
    df = pd.DataFrame(data).drop_duplicates().reset_index(drop=True)
    df = add_ru_en_labels(df, label_fields)
    if "item_en" not in df.columns:
        return pd.DataFrame()
    df = df[df["item_en"].apply(_good_label)].copy()
    return df.drop_duplicates().reset_index(drop=True)


print("Building/loading museums seed pool (clean v3)...")
museums_seed_df = load_or_build_pool("museums_seed_pool_clean_v3", lambda: build_museums_seed_pool())
print(f"museums_seed_df rows: {len(museums_seed_df)}")


def museums_seed_audit() -> Dict[str, Any]:
    df = museums_seed_df
    out = {"rows": int(len(df)) if df is not None else 0}
    if df is None or len(df) == 0:
        return out
    for f in ["type", "country", "city", "continent", "operator", "year_bucket"]:
        col = f"{f}_qid" if f != "year_bucket" else f
        out[f] = int(df[col].notna().sum()) if col in df.columns else 0
    return out


def _museum_group_candidates(df, fields: Sequence[str], min_n: int, max_n: Optional[int], max_rows: int = 1000):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    sub = df.copy()
    group_cols = []
    agg = {"n": ("item_qid", "nunique")}
    for field in fields:
        if field == "year_bucket":
            sub = sub[sub["year_bucket"].notna()]
            group_cols.append("year_bucket")
            for c in ["year_from", "year_to", "year_ru", "year_en"]:
                agg[c] = (c, "first")
        else:
            qcol = f"{field}_qid"
            encol = f"{field}_en"
            rucol = f"{field}_ru"
            if qcol not in sub.columns:
                return pd.DataFrame()
            sub = sub[sub[qcol].apply(_good_qid) & sub[encol].apply(_good_label) & sub[rucol].apply(_good_label)]
            group_cols.append(qcol)
            agg[encol] = (encol, "first")
            agg[rucol] = (rucol, "first")
    if len(sub) == 0:
        return pd.DataFrame()
    grp = sub.groupby(group_cols, dropna=False).agg(**agg).reset_index()
    grp = grp[grp["n"] >= int(min_n)]
    if max_n is not None:
        grp = grp[grp["n"] <= int(max_n)]
    grp = grp.sort_values(["n"], ascending=[True]).head(int(max_rows)).reset_index(drop=True)
    return grp


MUSEUM_DIRECT_TEMPLATES = [
    {"level": "L1", "template_id": "museums_l1_by_country", "family": "single_country", "fields": ["country"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "museums_l1_by_city", "family": "single_city", "fields": ["city"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "museums_l1_by_type", "family": "single_type", "fields": ["type"], "min_n": 5, "max_n": 250},
    {"level": "L1", "template_id": "museums_l1_by_continent", "family": "single_continent", "fields": ["continent"], "min_n": 5, "max_n": 250},

    {"level": "L2", "template_id": "museums_l2_country_type", "family": "country_type", "fields": ["country", "type"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "museums_l2_city_type", "family": "city_type", "fields": ["city", "type"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "museums_l2_continent_type", "family": "continent_type", "fields": ["continent", "type"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "museums_l2_country_period", "family": "country_period", "fields": ["country", "year_bucket"], "min_n": 5, "max_n": 200},
    {"level": "L2", "template_id": "museums_l2_operator_country", "family": "operator_country", "fields": ["operator", "country"], "min_n": 5, "max_n": 200},

    {"level": "L3", "template_id": "museums_l3_country_type_period", "family": "country_type_period", "fields": ["country", "type", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "museums_l3_city_type_period", "family": "city_type_period", "fields": ["city", "type", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "museums_l3_continent_type_period", "family": "continent_type_period", "fields": ["continent", "type", "year_bucket"], "min_n": 4, "max_n": 150},
    {"level": "L3", "template_id": "museums_l3_country_operator_period", "family": "country_operator_period", "fields": ["country", "operator", "year_bucket"], "min_n": 4, "max_n": 150},
]


def _make_museum_direct_spec(tpl: Dict[str, Any], row: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    level = tpl["level"]
    k = MUSEUMS_REQUESTED_COUNT[level]
    where_lines = []
    constraints = {"answer_type": "museum"}
    constraint_qids = {"answer_type": Q_MUSEUM}
    fragments_ru = []
    fragments_en = []
    for field in tpl["fields"]:
        if field == "year_bucket":
            y1 = _safe_int(row.get("year_from"))
            y2 = _safe_int(row.get("year_to"))
            if y1 is None or y2 is None:
                return None
            where_lines.extend(_museum_date_filter(y1, y2))
            constraints["inception_from"] = y1
            constraints["inception_to"] = y2
            fragments_ru.append(f"основанных в период {y1}–{y2} годов")
            fragments_en.append(f"founded between {y1} and {y2}")
            continue
        qid = row.get(f"{field}_qid")
        en = row.get(f"{field}_en")
        ru = row.get(f"{field}_ru")
        if not (_good_qid(qid) and _good_label(en) and _good_label(ru)):
            return None
        if field == "type" and _museum_type_is_bad(qid, en):
            return None
        if field == "type":
            ru = _museum_type_ru_label(str(en), str(ru))
        where_lines.extend(MUSEUM_FIELD_BUILDERS[field](qid))
        cname = MUSEUM_FIELD_CONSTRAINT_NAMES[field]
        constraints[cname] = str(en)
        constraint_qids[cname] = qid
        fr_ru, fr_en = _museum_fragment(field, str(en), str(ru))
        fragments_ru.append(fr_ru)
        fragments_en.append(fr_en)
    query_ru, query_en = _museum_query_text(k, fragments_ru, fragments_en)
    return {
        "complexity": level,
        "requested_count": k,
        "template_id": tpl["template_id"],
        "template_family": tpl["family"],
        "query_text_ru": query_ru,
        "query_text_en": query_en,
        "constraints": constraints,
        "constraint_qids": constraint_qids,
        "where_lines": _dedupe_lines(where_lines),
        "local_n": int(row.get("n", 0) or 0),
    }


def _museum_local_items(filters: Dict[str, Any], exclude_qids: Sequence[str] = ()) -> List[Tuple[str, str, str]]:
    df = museums_seed_df.copy()
    for field, qid in filters.items():
        if field in {"year_from", "year_to"}:
            continue
        if qid and f"{field}_qid" in df.columns:
            df = df[df[f"{field}_qid"] == qid]
    y1 = filters.get("year_from")
    y2 = filters.get("year_to")
    if y1 is not None and y2 is not None:
        df = df[df["year"].notna() & (df["year"] >= int(y1)) & (df["year"] <= int(y2))]
    ex = {q for q in exclude_qids if _good_qid(q)}
    if ex:
        df = df[~df["item_qid"].isin(ex)]
    items = []
    seen = set()
    for _, row in df.iterrows():
        q = row.get("item_qid")
        en = row.get("item_en")
        ru = row.get("item_ru") or en
        if _good_qid(q) and _good_label(en) and q not in seen:
            seen.add(q)
            items.append((q, ru, en))
    return items


def _museum_ref_for_field(field: str, qid: str, exclude: Sequence[str], rng: random.Random):
    if not _good_qid(qid):
        return None
    df = museums_seed_df[
        (museums_seed_df.get(f"{field}_qid") == qid)
        & museums_seed_df["item_qid"].apply(_good_qid)
        & museums_seed_df["item_en"].apply(_good_label)
    ].copy()
    if exclude:
        df = df[~df["item_qid"].isin(set(exclude))]
    if len(df) == 0:
        return None
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return {"qid": row["item_qid"], "en": row["item_en"], "ru": row.get("item_ru") or row["item_en"]}


def build_museum_direct_candidates(rng: random.Random) -> List[Dict[str, Any]]:
    specs = []
    for tpl in MUSEUM_DIRECT_TEMPLATES:
        grp = _museum_group_candidates(
            museums_seed_df,
            tpl["fields"],
            min_n=tpl["min_n"],
            max_n=tpl["max_n"],
            max_rows=900,
        )
        if len(grp) == 0:
            continue
        rows = grp.sample(frac=1, random_state=rng.randint(0, 10**9)).to_dict("records")
        for row in rows:
            spec = _make_museum_direct_spec(tpl, row)
            if spec:
                level = spec["complexity"]
                if spec.get("local_n", 0) >= MUSEUMS_REQUESTED_COUNT[level] + MUSEUMS_MIN_GOLD_HEADROOM and spec.get("local_n", 0) <= MUSEUMS_MAX_GOLD_BY_LEVEL[level]:
                    specs.append(spec)
    return specs


def _build_museum_bridge_candidates(rng: random.Random, max_candidates: int = 2400) -> List[Dict[str, Any]]:
    specs: List[Dict[str, Any]] = []
    if museums_seed_df is None or len(museums_seed_df) == 0:
        return specs
    required_cols = {
        "item_qid", "item_en", "type_qid", "type_en", "type_ru",
        "country_qid", "country_en", "country_ru", "city_qid", "city_en", "city_ru",
        "year_bucket", "year_from", "year_to",
    }
    if not required_cols.issubset(set(museums_seed_df.columns)):
        return specs
    df = museums_seed_df.copy()

    def emit(spec):
        if spec is None:
            return False
        local_n = len(_museum_local_items(spec["local_filters"], exclude_qids=spec.get("exclude_qids", [])))
        spec["local_n"] = local_n
        level = spec["complexity"]
        if local_n < MUSEUMS_REQUESTED_COUNT[level] + MUSEUMS_MIN_GOLD_HEADROOM or local_n > MUSEUMS_MAX_GOLD_BY_LEVEL[level]:
            return False
        spec.pop("local_filters", None)
        specs.append(spec)
        return len(specs) >= max_candidates

    # L4-A: same country as seed museum + type + period.
    rows = df[
        df["country_qid"].apply(_good_qid)
        & df["type_qid"].apply(_good_qid)
        & df["year_bucket"].notna()
        & df["country_en"].apply(_good_label)
        & df["type_en"].apply(_good_label)
        & ~df.apply(lambda r: _museum_type_is_bad(r.get("type_qid"), r.get("type_en")), axis=1)
    ].sample(frac=1, random_state=21)
    for _, row in rows.iterrows():
        seed = _museum_ref_for_field("country", row["country_qid"], [], rng)
        if not seed:
            continue
        y1, y2 = int(row["year_from"]), int(row["year_to"])
        k = MUSEUMS_REQUESTED_COUNT["L4"]
        fr_ru = [
            f"расположенных в той же стране, что и музей «{seed['ru']}»",
            f"относящихся к типу «{_museum_type_ru_label(row['type_en'], row['type_ru'])}»",
            f"основанных в период {y1}–{y2} годов",
        ]
        fr_en = [
            f"located in the same country as {seed['en']}",
            f'classified as "{row["type_en"]}"',
            f"founded between {y1} and {y2}",
        ]
        qru, qen = _museum_query_text(k, fr_ru, fr_en)
        stop = emit({
            "complexity": "L4", "requested_count": k,
            "template_id": "museums_l4_seed_country_type_period",
            "template_family": "seed_country_type_period",
            "query_text_ru": qru + f" Не включай музей «{seed['ru']}» в ответ.",
            "query_text_en": qen + f" Do not include the museum {seed['en']} in the answer.",
            "constraints": {
                "answer_type": "museum",
                "same_country_as_museum": seed["en"],
                "museum_type": row["type_en"],
                "inception_from": y1, "inception_to": y2,
                "exclude_museums": [seed["en"]],
            },
            "constraint_qids": {
                "answer_type": Q_MUSEUM,
                "same_country_as_museum": seed["qid"],
                "country": row["country_qid"],
                "museum_type": row["type_qid"],
                "exclude_museums": [seed["qid"]],
            },
            "where_lines": _dedupe_lines([
                f"BIND(wd:{seed['qid']} AS ?seedMuseum) .",
                "?seedMuseum wdt:P17 ?seedCountry .",
                "?item wdt:P17 ?seedCountry .",
                f"?item wdt:P31/wdt:P279* wd:{row['type_qid']} .",
                f"FILTER(?item != wd:{seed['qid']}) .",
                *_museum_date_filter(y1, y2),
            ]),
            "local_filters": {"country": row["country_qid"], "type": row["type_qid"], "year_from": y1, "year_to": y2},
            "exclude_qids": [seed["qid"]],
        })
        if stop:
            return specs

    # L4-B: same city/admin unit as seed museum + type.
    rows = df[
        df["city_qid"].apply(_good_qid)
        & df["type_qid"].apply(_good_qid)
        & df["city_en"].apply(_good_label)
        & df["type_en"].apply(_good_label)
        & ~df.apply(lambda r: _museum_type_is_bad(r.get("type_qid"), r.get("type_en")), axis=1)
    ].sample(frac=1, random_state=22)
    for _, row in rows.iterrows():
        seed = _museum_ref_for_field("city", row["city_qid"], [], rng)
        if not seed:
            continue
        k = MUSEUMS_REQUESTED_COUNT["L4"]
        fr_ru = [
            f"расположенных в той же административной единице или городе, что и музей «{seed['ru']}»",
            f"относящихся к типу «{_museum_type_ru_label(row['type_en'], row['type_ru'])}»",
        ]
        fr_en = [
            f"located in the same administrative unit or city as {seed['en']}",
            f'classified as "{row["type_en"]}"',
        ]
        qru, qen = _museum_query_text(k, fr_ru, fr_en)
        stop = emit({
            "complexity": "L4", "requested_count": k,
            "template_id": "museums_l4_seed_city_type",
            "template_family": "seed_city_type",
            "query_text_ru": qru + f" Не включай музей «{seed['ru']}» в ответ.",
            "query_text_en": qen + f" Do not include the museum {seed['en']} in the answer.",
            "constraints": {
                "answer_type": "museum",
                "same_location_as_museum": seed["en"],
                "museum_type": row["type_en"],
                "exclude_museums": [seed["en"]],
            },
            "constraint_qids": {
                "answer_type": Q_MUSEUM,
                "same_location_as_museum": seed["qid"],
                "located_in": row["city_qid"],
                "museum_type": row["type_qid"],
                "exclude_museums": [seed["qid"]],
            },
            "where_lines": _dedupe_lines([
                f"BIND(wd:{seed['qid']} AS ?seedMuseum) .",
                "?seedMuseum wdt:P131 ?seedLocation .",
                "?item wdt:P131+ ?seedLocation .",
                f"?item wdt:P31/wdt:P279* wd:{row['type_qid']} .",
                f"FILTER(?item != wd:{seed['qid']}) .",
            ]),
            "local_filters": {"city": row["city_qid"], "type": row["type_qid"]},
            "exclude_qids": [seed["qid"]],
        })
        if stop:
            return specs

    # L5-A: same country as seed A + same type as seed B + period.
    rows = df[
        df["country_qid"].apply(_good_qid)
        & df["type_qid"].apply(_good_qid)
        & df["year_bucket"].notna()
        & df["country_en"].apply(_good_label)
        & df["type_en"].apply(_good_label)
        & ~df.apply(lambda r: _museum_type_is_bad(r.get("type_qid"), r.get("type_en")), axis=1)
    ].sample(frac=1, random_state=23)
    for _, row in rows.iterrows():
        seed_a = _museum_ref_for_field("country", row["country_qid"], [], rng)
        seed_b = _museum_ref_for_field("type", row["type_qid"], [seed_a["qid"]] if seed_a else [], rng)
        if not (seed_a and seed_b):
            continue
        y1, y2 = int(row["year_from"]), int(row["year_to"])
        k = MUSEUMS_REQUESTED_COUNT["L5"]
        fr_ru = [
            f"расположенных в той же стране, что и музей «{seed_a['ru']}»",
            f"относящихся к тому же типу, что и музей «{seed_b['ru']}»",
            f"основанных в период {y1}–{y2} годов",
        ]
        fr_en = [
            f"located in the same country as {seed_a['en']}",
            f"of the same type as {seed_b['en']}",
            f"founded between {y1} and {y2}",
        ]
        qru, qen = _museum_query_text(k, fr_ru, fr_en)
        stop = emit({
            "complexity": "L5", "requested_count": k,
            "template_id": "museums_l5_seed_country_seed_type_period",
            "template_family": "two_seed_country_type_period",
            "query_text_ru": qru + f" Не включай музеи «{seed_a['ru']}» и «{seed_b['ru']}» в ответ.",
            "query_text_en": qen + f" Do not include the museums {seed_a['en']} and {seed_b['en']} in the answer.",
            "constraints": {
                "answer_type": "museum",
                "same_country_as_museum": seed_a["en"],
                "same_type_as_museum": seed_b["en"],
                "inception_from": y1, "inception_to": y2,
                "exclude_museums": [seed_a["en"], seed_b["en"]],
            },
            "constraint_qids": {
                "answer_type": Q_MUSEUM,
                "same_country_as_museum": seed_a["qid"],
                "country": row["country_qid"],
                "same_type_as_museum": seed_b["qid"],
                "museum_type": row["type_qid"],
                "exclude_museums": [seed_a["qid"], seed_b["qid"]],
            },
            "where_lines": _dedupe_lines([
                f"BIND(wd:{seed_a['qid']} AS ?seedMuseumA) .",
                f"BIND(wd:{seed_b['qid']} AS ?seedMuseumB) .",
                "?seedMuseumA wdt:P17 ?seedCountry .",
                "?item wdt:P17 ?seedCountry .",
                f"?seedMuseumB wdt:P31 wd:{row['type_qid']} .",
                f"?item wdt:P31/wdt:P279* wd:{row['type_qid']} .",
                f"FILTER(?item != wd:{seed_a['qid']} && ?item != wd:{seed_b['qid']}) .",
                *_museum_date_filter(y1, y2),
            ]),
            "local_filters": {"country": row["country_qid"], "type": row["type_qid"], "year_from": y1, "year_to": y2},
            "exclude_qids": [seed_a["qid"], seed_b["qid"]],
        })
        if stop:
            return specs

    # L5-B: same city/admin unit as seed A + same type as seed B + period.
    rows = df[
        df["city_qid"].apply(_good_qid)
        & df["type_qid"].apply(_good_qid)
        & df["year_bucket"].notna()
        & df["city_en"].apply(_good_label)
        & df["type_en"].apply(_good_label)
        & ~df.apply(lambda r: _museum_type_is_bad(r.get("type_qid"), r.get("type_en")), axis=1)
    ].sample(frac=1, random_state=24)
    for _, row in rows.iterrows():
        seed_a = _museum_ref_for_field("city", row["city_qid"], [], rng)
        seed_b = _museum_ref_for_field("type", row["type_qid"], [seed_a["qid"]] if seed_a else [], rng)
        if not (seed_a and seed_b):
            continue
        y1, y2 = int(row["year_from"]), int(row["year_to"])
        k = MUSEUMS_REQUESTED_COUNT["L5"]
        fr_ru = [
            f"расположенных в той же административной единице или городе, что и музей «{seed_a['ru']}»",
            f"относящихся к тому же типу, что и музей «{seed_b['ru']}»",
            f"основанных в период {y1}–{y2} годов",
        ]
        fr_en = [
            f"located in the same administrative unit or city as {seed_a['en']}",
            f"of the same type as {seed_b['en']}",
            f"founded between {y1} and {y2}",
        ]
        qru, qen = _museum_query_text(k, fr_ru, fr_en)
        stop = emit({
            "complexity": "L5", "requested_count": k,
            "template_id": "museums_l5_seed_city_seed_type_period",
            "template_family": "two_seed_city_type_period",
            "query_text_ru": qru + f" Не включай музеи «{seed_a['ru']}» и «{seed_b['ru']}» в ответ.",
            "query_text_en": qen + f" Do not include the museums {seed_a['en']} and {seed_b['en']} in the answer.",
            "constraints": {
                "answer_type": "museum",
                "same_location_as_museum": seed_a["en"],
                "same_type_as_museum": seed_b["en"],
                "inception_from": y1, "inception_to": y2,
                "exclude_museums": [seed_a["en"], seed_b["en"]],
            },
            "constraint_qids": {
                "answer_type": Q_MUSEUM,
                "same_location_as_museum": seed_a["qid"],
                "located_in": row["city_qid"],
                "same_type_as_museum": seed_b["qid"],
                "museum_type": row["type_qid"],
                "exclude_museums": [seed_a["qid"], seed_b["qid"]],
            },
            "where_lines": _dedupe_lines([
                f"BIND(wd:{seed_a['qid']} AS ?seedMuseumA) .",
                f"BIND(wd:{seed_b['qid']} AS ?seedMuseumB) .",
                "?seedMuseumA wdt:P131 ?seedLocation .",
                "?item wdt:P131+ ?seedLocation .",
                f"?seedMuseumB wdt:P31 wd:{row['type_qid']} .",
                f"?item wdt:P31/wdt:P279* wd:{row['type_qid']} .",
                f"FILTER(?item != wd:{seed_a['qid']} && ?item != wd:{seed_b['qid']}) .",
                *_museum_date_filter(y1, y2),
            ]),
            "local_filters": {"city": row["city_qid"], "type": row["type_qid"], "year_from": y1, "year_to": y2},
            "exclude_qids": [seed_a["qid"], seed_b["qid"]],
        })
        if stop:
            return specs

    return specs


def build_museum_candidate_queue(seed: int = 20260528) -> Dict[str, List[Dict[str, Any]]]:
    rng = random.Random(seed)
    all_specs = build_museum_direct_candidates(rng) + _build_museum_bridge_candidates(rng)
    by_level = {level: [] for level in MUSEUMS_TARGET_PER_LEVEL}
    seen = set()
    for spec in all_specs:
        c = spec.get("constraints", {})
        if _museum_type_is_bad((spec.get("constraint_qids") or {}).get("museum_type"), c.get("museum_type")):
            continue
        key = json.dumps({"level": spec["complexity"], "template_id": spec["template_id"], "constraints": spec["constraints"]}, ensure_ascii=False, sort_keys=True)
        if key in seen:
            continue
        seen.add(key)
        by_level.setdefault(spec["complexity"], []).append(spec)
    for level in by_level:
        rng.shuffle(by_level[level])
    return by_level


def museums_candidate_audit(queue: Optional[Dict[str, List[Dict[str, Any]]]] = None) -> pd.DataFrame:
    if queue is None:
        queue = build_museum_candidate_queue()
    rows = []
    for level, specs in queue.items():
        fam = Counter(s["template_family"] for s in specs)
        rows.append({"complexity": level, "candidates": len(specs), "families": dict(fam)})
    return pd.DataFrame(rows)


Building/loading museums seed pool (clean v3)...
museums_seed_df rows: 7499


## Generation

In [4]:

# ============================================================
# Generate / resume museums JSONL incrementally
# ============================================================
def generate_museums_dataset(
    output_path: Path = MUSEUMS_OUTPUT_PATH,
    target_per_level: Dict[str, int] = MUSEUMS_TARGET_PER_LEVEL,
    seed: int = 20260528,
    require_complete_gold: bool = True,
    reset_output: bool = False,
) -> List[Dict[str, Any]]:
    rng = random.Random(seed)
    if reset_output:
        _archive_existing_jsonl(output_path, suffix="pre_v3")
    existing = _read_jsonl(output_path)
    if existing:
        existing_problems = validate_jsonl_exact_format(output_path)
        if len(existing_problems) > 0:
            raise AssertionError(f"Existing JSONL has schema/quality problems; rerun with reset_output=True. Preview: {existing_problems.head(10).to_dict('records')}")
    seen_keys = {_record_key(r) for r in existing}
    counts = Counter(r.get("complexity") for r in existing if r.get("domain") == MUSEUMS_DOMAIN)
    next_idx = defaultdict(lambda: 1)
    for r in existing:
        if r.get("domain") != MUSEUMS_DOMAIN:
            continue
        m = re.search(r"_(l\d)_(\d{4})$", str(r.get("id", "")))
        if m:
            level = m.group(1).upper()
            next_idx[level] = max(next_idx[level], int(m.group(2)) + 1)

    queue = build_museum_candidate_queue(seed=seed)
    audit_skips = []
    total_target = sum(target_per_level.values())
    current_total = sum(min(counts.get(level, 0), target) for level, target in target_per_level.items())
    pbar = tqdm(total=total_target, initial=current_total, desc="museums total")

    for level, target in target_per_level.items():
        accepted_for_level = int(counts.get(level, 0))
        attempts = 0
        specs = queue.get(level, [])
        rng.shuffle(specs)
        spec_i = 0
        while accepted_for_level < int(target) and attempts < MUSEUMS_MAX_ATTEMPTS_PER_LEVEL:
            attempts += 1
            if not specs:
                audit_skips.append({"level": level, "reason": "no_candidate_specs"})
                break
            spec = specs[spec_i % len(specs)]
            spec_i += 1
            proposed_key = json.dumps({"domain": MUSEUMS_DOMAIN, "complexity": level, "template_id": spec["template_id"], "constraints": spec["constraints"]}, ensure_ascii=False, sort_keys=True)
            if proposed_key in seen_keys:
                continue
            try:
                ex = _finalize_museum_spec(spec, next_idx[level], require_complete=require_complete_gold)
            except Exception as e:
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": "exception", "error": str(e)[:500]})
                continue
            if ex is None:
                audit_skips.append({"level": level, "template_id": spec.get("template_id"), "reason": "gold_rejected_or_incomplete", "constraints": spec.get("constraints")})
                continue
            rec = _example_to_record(ex)
            if not _schema_is_exact(rec):
                raise AssertionError("Generated record field order does not match BenchmarkExample")
            key = _record_key(rec)
            if key in seen_keys:
                continue
            _append_jsonl(output_path, rec)
            existing.append(rec)
            seen_keys.add(key)
            seen_keys.add(proposed_key)
            next_idx[level] += 1
            counts[level] += 1
            accepted_for_level += 1
            pbar.update(1)
        if accepted_for_level < int(target):
            print(f"[WARN] museums {level}: accepted {accepted_for_level}/{target}; inspect audit for skipped candidates.")
    pbar.close()

    audit = {
        "domain": MUSEUMS_DOMAIN,
        "output_path": str(output_path),
        "target_per_level": target_per_level,
        "counts_by_complexity": dict(Counter(r.get("complexity") for r in existing if r.get("domain") == MUSEUMS_DOMAIN)),
        "seed_pool": museums_seed_audit(),
        "candidate_counts": {lvl: len(v) for lvl, v in queue.items()},
        "skipped_count": len(audit_skips),
        "skipped_preview": audit_skips[-200:],
        "updated_at": _now_iso(),
        "format_fields": BENCHMARK_FIELD_ORDER,
    }
    format_problems = validate_jsonl_exact_format(output_path)
    if len(format_problems) > 0:
        raise AssertionError(f"Generated JSONL format mismatch: {format_problems.head(10).to_dict('records')}")
    _write_json(MUSEUMS_AUDIT_PATH, audit)
    print(f"saved incrementally: {output_path}")
    print(f"audit: {MUSEUMS_AUDIT_PATH}")
    return existing


def smoke_test_museums(per_level: int = 1, seed: int = 42) -> pd.DataFrame:
    """Runs a tiny WDQS-backed generation pass without writing JSONL."""
    queue = build_museum_candidate_queue(seed=seed)
    rows = []
    for level in MUSEUMS_TARGET_PER_LEVEL:
        ok = 0
        for spec in queue.get(level, [])[:80]:
            if ok >= per_level:
                break
            try:
                ex = _finalize_museum_spec(spec, idx=ok + 1, require_complete=True)
            except Exception as e:
                rows.append({"complexity": level, "ok": False, "template_id": spec.get("template_id"), "error": str(e)[:200]})
                continue
            if ex is not None:
                rows.append({"complexity": level, "ok": True, "template_id": ex.template_id, "gold": len(ex.gold_answer_qids), "query_en": ex.query_text_en})
                ok += 1
    return pd.DataFrame(rows)


# Run the full generator by default when this final cell is executed.
# Set RUN_MUSEUMS_GENERATION = False before running this cell if you only want to inspect/debug.
RUN_MUSEUMS_GENERATION = True
# The previous v1/v2 runs may contain records with noisy museum_type classes or old metadata.
# Keep this True for a clean v3 regeneration; the old JSONL is archived, not deleted.
RESET_MUSEUMS_OUTPUT = True
if RUN_MUSEUMS_GENERATION:
    museums_records = generate_museums_dataset(reset_output=RESET_MUSEUMS_OUTPUT)
else:
    print("Generation skipped. Use smoke_test_museums() or generate_museums_dataset() manually.")


museums total: 100%|██████████| 110/110 [2:21:47<00:00, 77.34s/it]  

saved incrementally: out_wikidata_benchmark/domain_outputs/museums.jsonl
audit: out_wikidata_benchmark/domain_outputs/museums.audit.json
